In [0]:
# =====================================
# NOTEBOOK : NB_SILVER_CLAIM
# LAYER    : SILVER
# TARGET   : ODS_CLAIM
# =====================================


In [0]:
%run ../COMMON/NB_CONFIG

In [0]:
%run ../COMMON/NB_CONNECTION

In [0]:
%run ../COMMON/NB_UTILS

In [0]:
claim_df = spark.read.jdbc(
    url=srcJdbcUrl,
    table="BRONZE.CLAIM",
    properties=connectionProperties
)

print("CLAIM COUNT :", claim_df.count())

In [0]:
null_count = claim_df.filter(F.col("CLAIM_ID").isNull()).count()
if null_count > 0:
    raise Exception(f"NULL CLAIM_ID FOUND : {null_count}")

In [0]:
ods_claim_df = claim_df.select(
    "CLAIM_ID","MEMBER_ID","PROVIDER_ID",
    "CLAIM_DATE","RECEIVED_DATE","PROCESSED_DATE","PAID_DATE",
    "CLAIM_AMOUNT","CLAIM_STATUS","DIAGNOSIS_CODE","PROCEDURE_CODE",
    "SRC_CREATED_DATETIME"
)

In [0]:
ods_claim_df = (
    ods_claim_df
    .withColumn("CLAIM_YEAR", F.year("CLAIM_DATE"))
    .withColumn("CLAIM_MONTH", F.month("CLAIM_DATE"))
    .withColumn("CLAIM_AGE_DAYS", F.datediff("RECEIVED_DATE","CLAIM_DATE"))
    .withColumn("PROCESSING_TIME_DAYS", F.datediff("PROCESSED_DATE","RECEIVED_DATE"))
    .withColumn("PAYMENT_LAG_DAYS", F.datediff("PAID_DATE","PROCESSED_DATE"))
    .withColumn("CLAIM_BUCKET",
        F.when(F.col("CLAIM_AMOUNT") < 1000, "<1000")
         .when(F.col("CLAIM_AMOUNT") < 5000, "1000-5000")
         .otherwise(">5000")
    )
    .withColumn("STATUS_FLAG",
        F.when(F.col("CLAIM_STATUS").isin("Submitted","Pending"),"OPEN")
         .when(F.col("CLAIM_STATUS").isin("Paid","Denied"),"CLOSED")
         .otherwise("UNKNOWN")
    )
)

In [0]:
ods_claim_df = ods_claim_df.fillna({
    "CLAIM_STATUS":"UNKNOWN",
    "CLAIM_AMOUNT":0
})

In [0]:
window_spec = Window.partitionBy("CLAIM_ID").orderBy(F.col("SRC_CREATED_DATETIME").desc())

ods_claim_df = (
    ods_claim_df
    .withColumn("RN", F.row_number().over(window_spec))
    .filter(F.col("RN") == 1)
    .drop("RN")
)

In [0]:
ods_claim_df = ods_claim_df.withColumn(
    "HASH_VALUE",
    F.sha2(
        F.concat_ws("|",
            "MEMBER_ID","PROVIDER_ID","CLAIM_DATE","RECEIVED_DATE",
            "PROCESSED_DATE","PAID_DATE","CLAIM_AMOUNT","CLAIM_STATUS"
        ),256
    )
)

In [0]:
ods_claim_df = (
    ods_claim_df
    .withColumn("LOAD_DATE", F.current_timestamp())
    .withColumn("IS_CURRENT", F.lit("Y"))
    .withColumn("EFFECTIVE_DATE", F.current_timestamp())
    .withColumn("END_DATE", F.lit("9999-12-31").cast("timestamp"))
)

In [0]:
target_count = spark.table(ods_claim_table).count()
print(f"TARGET RECORD COUNT : {target_count}")

if target_count == 0:
    # Initial full load
    ods_claim_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(ods_claim_table)

    print("INITIAL LOAD COMPLETED")
    dbutils.notebook.exit("Initial load done")

else:
    print("SCD2 LOAD STARTED")
    target_df = spark.table(ods_claim_table)
    current_df = target_df.filter(F.col("IS_CURRENT") == "Y")
    print(f"CURRENT ACTIVE RECORDS : {current_df.count()}")

In [0]:
changed_df = (
    ods_claim_df.alias("s")
    .join(current_df.alias("t"), "CLAIM_ID")
    .filter(F.col("s.HASH_VALUE") != F.col("t.HASH_VALUE"))
    .select("s.*")
)

In [0]:
new_claim_df = (
    ods_claim_df.alias("s")
    .join(current_df.alias("t"), "CLAIM_ID", "left_anti")
)

In [0]:
delta_table = DeltaTable.forName(spark, ods_claim_table)

expire_df = changed_df.select("CLAIM_ID").distinct()

delta_table.alias("t").merge(
    expire_df.alias("s"),
    "t.CLAIM_ID = s.CLAIM_ID AND t.IS_CURRENT = 'Y'"
).whenMatchedUpdate(
    set={
        "IS_CURRENT": "'N'",
        "END_DATE": "current_timestamp()"
    }
).execute()

In [0]:
changed_insert_df = (
    changed_df
    .withColumn("LOAD_DATE", F.current_timestamp())
    .withColumn("IS_CURRENT", F.lit("Y"))
    .withColumn("EFFECTIVE_DATE", F.current_timestamp())
    .withColumn("END_DATE", F.lit("9999-12-31").cast("timestamp"))
)

changed_insert_df.write.format("delta").mode("append").saveAsTable(ods_claim_table)

In [0]:
new_claim_df = (
    new_claim_df
    .withColumn("LOAD_DATE", F.current_timestamp())
    .withColumn("IS_CURRENT", F.lit("Y"))
    .withColumn("EFFECTIVE_DATE", F.current_timestamp())
    .withColumn("END_DATE", F.lit("9999-12-31").cast("timestamp"))
)

new_claim_df.write.format("delta").mode("append").saveAsTable(ods_claim_table)

In [0]:
display(
    spark.sql(f"""
    SELECT *
    FROM {ods_claim_table}
    ORDER BY CLAIM_ID
    """)
)